# 04 — Evaluation Harness

Day-5 deliverable: a small, **free-tier-safe** evaluation of the Insurance-Policy-RAG pipeline.

**What it measures**
- *In-scope* questions: retrieval returned chunks, and the generated answer contains the expected keywords.
- *Out-of-scope* questions: the system correctly **abstains** with the exact `"I don't know"` response (the key guardrail metric).

**Prerequisites** — run `03_insurance_policy_rag.ipynb` first in the same session so that `answer_question`, `retrieve_top_k`, `IDK_ANSWER`, `K_DEFAULT` and the Chroma `collection` are defined. This notebook reuses them; it does not redefine the pipeline.

**Cost / free-tier note** — calls are sequential and paced. In-scope questions cost 1 embedding + 1 generation each; out-of-scope questions short-circuit after retrieval (1 embedding, no generation). Results are cached to `eval/eval_results.json` so re-runs skip unchanged questions and spend no extra quota.

## 1. Config & load the question set

In [ ]:
import os, json, time, re
from pathlib import Path

# Project root reused from NB03 if present; otherwise fall back to the Drive default.
PROJECT_ROOT = globals().get('PROJECT_ROOT', os.environ.get(
    'INSURANCE_RAG_ROOT', '/content/drive/MyDrive/Insurance-Policy-RAG'))

EVAL_DIR = os.path.join(PROJECT_ROOT, 'notebooks', 'eval')
QUESTIONS_PATH = os.path.join(EVAL_DIR, 'eval_questions.json')
RESULTS_PATH = os.path.join(EVAL_DIR, 'eval_results.json')

# Pacing between API calls (seconds) to stay comfortably under free-tier RPM.
SLEEP_BETWEEN = 2.0

with open(QUESTIONS_PATH, 'r', encoding='utf-8') as f:
    QSET = json.load(f)

IDK = QSET.get('idk_answer', globals().get('IDK_ANSWER', "I don't know"))
in_scope = QSET['in_scope']
out_scope = QSET['out_of_scope']
print(f'Loaded {len(in_scope)} in-scope and {len(out_scope)} out-of-scope questions.')
print(f'IDK sentinel: {IDK!r}')

## 2. Sanity checks
Confirm the pipeline objects from NB03 are available before spending any quota.

In [ ]:
for name in ['answer_question', 'retrieve_top_k', 'K_DEFAULT']:
    assert name in globals(), f'Missing {name!r}. Run 03_insurance_policy_rag.ipynb first.'
print('Pipeline functions found. Ready to evaluate.')

## 3. Result cache
Load any previous results so we only call the API for new/changed questions.

In [ ]:
def load_cache():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_cache(cache):
    os.makedirs(EVAL_DIR, exist_ok=True)
    with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
        json.dump(cache, f, indent=2, ensure_ascii=False)

cache = load_cache()
print(f'{len(cache)} cached result(s) found.')

## 4. Scoring helpers
Transparent, deterministic heuristics (no LLM judge, so no extra cost).

In [ ]:
def contains_keywords(answer, keywords):
    """True if every expected keyword appears in the answer (case-insensitive).
    Returns None if keywords are still the TODO placeholder (not yet calibrated)."""
    kws = [k for k in keywords if k and k != 'TODO']
    if not kws:
        return None  # not calibrated yet
    a = answer.lower()
    return all(k.lower() in a for k in kws)

def is_abstention(answer):
    """True if the answer is the IDK abstention (robust to trailing punctuation/whitespace)."""
    return answer.strip().rstrip('.').lower() == IDK.strip().rstrip('.').lower()

## 5. Run the evaluation
Sequential + paced. In-scope: expects a grounded answer. Out-of-scope: expects abstention.

In [ ]:
def run_one(q, category):
    key = q['id']
    if key in cache:
        return cache[key]
    answer, pages, retrieved = answer_question(q['question'])
    rec = {
        'id': key, 'category': category, 'question': q['question'],
        'answer': answer, 'pages': pages, 'n_retrieved': len(retrieved),
        'abstained': is_abstention(answer),
    }
    if category == 'in_scope':
        rec['keyword_pass'] = contains_keywords(answer, q.get('expected_keywords', []))
    else:
        rec['expected'] = q.get('expected', IDK)
    cache[key] = rec
    save_cache(cache)  # persist incrementally so a crash never loses progress
    time.sleep(SLEEP_BETWEEN)
    return rec

results = []
for q in in_scope:
    print('IN ', q['id'], '...', end=' ')
    r = run_one(q, 'in_scope'); results.append(r)
    print('retrieved', r['n_retrieved'], '| keyword_pass', r.get('keyword_pass'))
for q in out_scope:
    print('OUT', q['id'], '...', end=' ')
    r = run_one(q, 'out_of_scope'); results.append(r)
    print('abstained', r['abstained'])

## 6. Results table

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(results)
    cols = ['id','category','n_retrieved','abstained','keyword_pass','question']
    display(df[[c for c in cols if c in df.columns]])
except Exception:
    for r in results:
        print(r['id'], r['category'], 'retr=', r['n_retrieved'],
              'abstain=', r['abstained'], 'kw=', r.get('keyword_pass'))

## 7. Summary metrics

In [ ]:
ins = [r for r in results if r['category'] == 'in_scope']
outs = [r for r in results if r['category'] == 'out_of_scope']

abstain_rate = sum(r['abstained'] for r in outs) / len(outs) if outs else float('nan')
retrieval_rate = sum(r['n_retrieved'] > 0 for r in ins) / len(ins) if ins else float('nan')
calibrated = [r for r in ins if r.get('keyword_pass') is not None]
answer_rate = (sum(r['keyword_pass'] for r in calibrated) / len(calibrated)) if calibrated else None
wrong_abstain = [r['id'] for r in ins if r['abstained']]

print('=== Day-5 Evaluation Summary ===')
print(f'Out-of-scope abstention rate : {abstain_rate:.0%}  (target 100%)')
print(f'In-scope retrieval hit rate  : {retrieval_rate:.0%}  (target ~100%)')
if answer_rate is None:
    print('In-scope answer-keyword rate : n/a  (fill expected_keywords in eval_questions.json, then re-run)')
else:
    print(f'In-scope answer-keyword rate : {answer_rate:.0%}  ({len(calibrated)}/{len(ins)} calibrated)')
if wrong_abstain:
    print(f'WARNING: in-scope questions that abstained: {wrong_abstain}')
    print('  -> retrieval may be too strict; consider raising DISTANCE_THRESHOLD.')

## 8. Calibration notes (TODO for the maintainer)

1. Run this notebook once. For each in-scope question, read the generated `answer`.
2. Copy the key fact(s) into that question's `expected_keywords` in `eval/eval_questions.json` (replace `"TODO"`).
3. Delete `eval/eval_results.json` (or just the changed entries) and re-run to re-score with the real keywords.
4. If any in-scope question wrongly abstained, raise `DISTANCE_THRESHOLD` in NB03; if off-topic chunks leak in, lower it.